In [1]:
!pip install /kaggle/input/pip-install-lifelines/autograd-1.7.0-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/autograd-gamma-0.5.0.tar.gz
!pip install /kaggle/input/pip-install-lifelines/interface_meta-1.3.0-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/formulaic-1.0.2-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/lifelines-0.30.0-py3-none-any.whl

Processing /kaggle/input/pip-install-lifelines/autograd-1.7.0-py3-none-any.whl
autograd is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
Processing /kaggle/input/pip-install-lifelines/autograd-gamma-0.5.0.tar.gz
  Preparing metadata (setup.py) ... done
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4031 sha256=d170a307d5cdf1c0801f318e08ed2bf7d96cebef85084e3786e201b6657dee0e
  Stored in directory: /root/.cache/pip/wheels/6b/b5/e0/4c79e15c0b5f2c15ecf613c720bb20daab20a666eb67135155
Successfully built autograd-gamma
Processing /kaggle/input/pip-install-lifelines/interface_meta-1.3.0-py3-none-any.whl
Processing /kaggle/input/pip-install-lifelines/formulaic-1.0.2-py3-none-any.whl
Processing /kaggle/input/pip-install-lifelines/lifelines-0.30.0-py3-none-any.whl


In [2]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats
import tensorflow_decision_forests as tfdf

In [3]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from lifelines import KaplanMeierFitter

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [4]:
def calculate_survival_probabilities(df, time_col, event_col):
    kmf = KaplanMeierFitter()
    kmf.fit(df[time_col], df[event_col])
    return kmf.survival_function_at_times(df[time_col]).values

def preprocess_survival_data(df, time_col='efs_time', event_col='efs'):
    df['target'] = calculate_survival_probabilities(df, time_col, event_col)
    df.loc[df[event_col] == 0, 'target'] -= 0.25
    return df

In [5]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['ID'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])

In [6]:
encoder = LabelEncoder()
normalize = MinMaxScaler()

for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]] = train[i[0]].fillna('Unknown')
        train[i[0]]=normalize.fit_transform(encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1)).reshape(-1,1))
    else:
        if i[0]!='efs' or i[0]!='efs_time':
            train[i[0]] = train[i[0]].fillna(0)
            train[i[0]]=normalize.fit_transform(numpy.array(train[i[0]]).reshape(-1,1))
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]] = test[i[0]].fillna('Unknown')
        test[i[0]]=normalize.fit_transform(numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1))).reshape(-1,1))
    else:
        if i[0]!='efs' or i[0]!='efs_time':
            test[i[0]] = test[i[0]].fillna(0)
            test[i[0]]=normalize.fit_transform(numpy.array(test[i[0]]).reshape(-1,1))

In [7]:
train = preprocess_survival_data(train)

train_x = train.drop(columns=['efs', 'efs_time', 'target'])
train_y = train['target']
train_x.head()

,dri_score,psych_disturb,cyto_score,diabetes,hla_match_c_high,hla_high_res_8,tbi_status,arrhythmia,hla_low_res_6,graft_type,...,karnofsky_score,hepatic_mild,tce_div_match,donor_related,melphalan_dose,hla_low_res_8,cardiac,hla_match_drb1_high,pulm_moderate,hla_low_res_10
0,0.636364,0.0,1.000000,0.0,0.0,0.0,0.000000,0.0,1.0,0.0,...,0.9,0.0,1.00,1.000000,0.5,1.0,0.0,1.0,0.0,1.0
1,0.181818,0.0,0.142857,0.0,1.0,1.0,0.857143,0.0,1.0,1.0,...,0.9,0.0,0.75,0.333333,0.5,1.0,0.0,1.0,1.0,1.0
2,0.636364,0.0,1.000000,0.0,1.0,1.0,0.000000,0.0,1.0,0.0,...,0.9,0.0,0.75,0.333333,0.5,1.0,0.0,1.0,0.0,1.0
3,0.000000,0.0,0.142857,0.0,1.0,1.0,0.000000,0.0,1.0,0.0,...,0.9,1.0,0.75,1.000000,0.5,1.0,0.0,1.0,0.0,1.0
4,0.000000,0.0,1.000000,0.0,1.0,1.0,0.000000,0.0,1.0,1.0,...,0.9,0.0,0.75,0.333333,0.0,1.0,0.0,1.0,0.0,1.0


In [8]:
Fold=5
kf = KFold(n_splits=Fold)

cat_pre_train = numpy.zeros(len(train))
cat_pre_test = numpy.zeros(len(test))

for i, (train_index, test_index) in enumerate(kf.split(train)):

    print(f"FOLD: {i}")

    x_fold=train_x[train_index[0]: train_index[len(train_index)-1]]
    y_fold=train_y[train_index[0]: train_index[len(train_index)-1]]

    x_test_fold=train_x[test_index[0]: test_index[len(test_index)-1]]
    y_test_fold=train_y[test_index[0]: test_index[len(test_index)-1]]

    cat_features = list(train_x.select_dtypes(include=['object', 'category']).columns)
    
    train_pool = Pool(x_fold, y_fold, cat_features=cat_features)
    val_pool = Pool(x_test_fold, y_test_fold, cat_features=cat_features)
    
    #weight_ = weight[train_index[0]: train_index[len(train_index)-1]]
    
    catboost = CatBoostRegressor( grow_policy='Depthwise',
                                 iterations=1000,
                                 eval_metric='RMSE',
                                 random_state=0,
                                 verbose=100)
    
    catboost.fit(train_pool, eval_set=(val_pool), early_stopping_rounds=100)

    cat_pre_train+=catboost.predict(train_x)
    cat_pre_test+=catboost.predict(test)

    print(catboost.score(x_test_fold, y_test_fold))

FOLD: 0
Learning rate set to 0.08333
0:	learn: 0.2806259	test: 0.2795300	best: 0.2795300 (0)	total: 73.1ms	remaining: 1m 13s
100:	learn: 0.2423662	test: 0.2538549	best: 0.2538549 (100)	total: 1.39s	remaining: 12.4s
200:	learn: 0.2222327	test: 0.2525792	best: 0.2525258 (162)	total: 2.66s	remaining: 10.6s
300:	learn: 0.2064932	test: 0.2530214	best: 0.2524888 (219)	total: 3.9s	remaining: 9.06s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.2524888037
bestIteration = 219

Shrink model to first 220 iterations.
0.19981179213781164
FOLD: 1
Learning rate set to 0.086301
0:	learn: 0.2803304	test: 0.2803600	best: 0.2803600 (0)	total: 16.1ms	remaining: 16.1s
100:	learn: 0.2434702	test: 0.2447950	best: 0.2447950 (100)	total: 1.57s	remaining: 14s
200:	learn: 0.2250673	test: 0.2255563	best: 0.2255563 (200)	total: 3.4s	remaining: 13.5s
300:	learn: 0.2111501	test: 0.2112618	best: 0.2112618 (300)	total: 4.96s	remaining: 11.5s
400:	learn: 0.1996532	test: 0.1996725	best: 0.1996725 (

In [9]:
id = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv')['ID']
test_predictions = (cat_pre_test/Fold)
#((RandomF_pre_test/Fold)+(LGBM_pre_test/Fold)+(xgb_pre_test/Fold)+(cat_pre_test/Fold))/4
#model.predict(new_test_data)
#((RandomF_pre_test/10)+(LGBM_pre_test/5)+(xgb_pre_test/5)+(cat_pre_test/5))/4
test_predictions

array([0.29024746, 0.80318875, 0.53206735])

In [10]:
submission = pandas.DataFrame({
    'ID': id.values,
    'prediction': test_predictions,
})
submission

,ID,prediction
0,28800,0.290247
1,28801,0.803189
2,28802,0.532067


In [11]:
submission.to_csv('submission.csv', index = False)